**Navigation** : [carnet source Python](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb) | [Index de la série](README.md)

# Chaînes-04 — L'indiscernabilité et le signe spin-statistique [jumeau C#]

Quatrième carnet de la série [Chaînes](README.md) — distillation du chapitre 43 du compendium Chain Theory
(Wissner-Gross, 2026, preprint non relu) : *Indistinguishability and the spin-statistics sign*.
Le fil : après l'acte de la mesure (Chaînes-03), ce que la construction dit des corps qu'on ne
peut pas distinguer — deux copies d'un site lues comme un seul qudit de dimension $d^2$, le swap
lavé par le twirl du groupe d'identification, et le signe de l'échange lu à trois faces.

Ce carnet est le **jumeau C#** du carnet Python de même titre : mêmes entrées, mêmes valeurs,
portées ici en **arithmétique exacte** (`Rationnel` sur `BigInteger`, BCL seule, aucun NuGet).
Renforcement du port : chaque valeur mesurée est portée par une **assertion exécutable** qui lève
sur divergence — les impressions seules ne suffisent pas.

[Version Python](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb)

Patron de la série : **Mesuré** (rejoué par le code de ce carnet) · **Cité** (toute la chaîne
dérivative de la source, jamais rejouée ici) · **Ouvert** (ce que la source ne prouve pas,
listé ligne à ligne, jamais maquillé en résultat établi).

In [1]:
#nullable enable
using System;
using System.Globalization;
using System.Numerics;

// COUTIL du carnet : Rationnel sur BigInteger (arithmetique exacte). Aucune egalite flottante,
// aucune racine substituee par un double -- la spec du port l'exige et le carnet le tient.

var demoPorte = new Rationnel(2, 16);
Console.WriteLine($"2/16 construit puis normalise : {demoPorte} (valeur brute {demoPorte.NumerateurBrut}/{demoPorte.DenominateurBrut})");
Console.WriteLine($"1/2 + 1/3 = {new Rationnel(1, 2) + new Rationnel(1, 3)}");
Console.WriteLine($"1/2 * 1/3 = {new Rationnel(1, 2) * new Rationnel(1, 3)}");

bool egaliteExacte = new Rationnel(1, 2).Equals(new Rationnel(2, 4));
Console.WriteLine($"1/2 egale 2/4 (egalite exacte) : {egaliteExacte}");
if (!egaliteExacte)
    throw new InvalidOperationException("ECART : 1/2 n'est pas egal a 2/4 en arithmetique rationnelle exacte.");

// Rationnel exact local : numerateur et denominateur en BigInteger, reduction par PGCD,
// denominateur strictement positif. Aucune comparaison flottante, aucune dependance externe.
public sealed class Rationnel : IEquatable<Rationnel>
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }
    public BigInteger NumerateurBrut { get; }
    public BigInteger DenominateurBrut { get; }

    public Rationnel(BigInteger numerateur, BigInteger denominateur)
    {
        if (denominateur.IsZero)
            throw new DivideByZeroException("Denominateur nul : rationnel indefini.");

        NumerateurBrut = numerateur;
        DenominateurBrut = denominateur;

        BigInteger pgcd = BigInteger.GreatestCommonDivisor(BigInteger.Abs(numerateur), BigInteger.Abs(denominateur));
        if (pgcd.IsZero)
            pgcd = BigInteger.One;

        Numerateur = numerateur / pgcd;
        Denominateur = denominateur / pgcd;

        if (Denominateur.Sign < 0)
        {
            Numerateur = -Numerateur;
            Denominateur = -Denominateur;
        }
    }

    public bool EstEntier => Denominateur.IsOne;

    public Rationnel(long entier) : this(new BigInteger(entier), BigInteger.One) { }

    public static Rationnel operator +(Rationnel a, Rationnel b)
        => new Rationnel(a.Numerateur * b.Denominateur + b.Numerateur * a.Denominateur,
                         a.Denominateur * b.Denominateur);

    public static Rationnel operator -(Rationnel a, Rationnel b)
        => new Rationnel(a.Numerateur * b.Denominateur - b.Numerateur * a.Denominateur,
                         a.Denominateur * b.Denominateur);

    public static Rationnel operator *(Rationnel a, Rationnel b)
        => new Rationnel(a.Numerateur * b.Numerateur, a.Denominateur * b.Denominateur);

    public static Rationnel operator /(Rationnel a, Rationnel b)
        => new Rationnel(a.Numerateur * b.Denominateur, a.Denominateur * b.Numerateur);

    public static bool operator ==(Rationnel? a, Rationnel? b)
        => a is null ? b is null : a.Equals(b);

    public static bool operator !=(Rationnel? a, Rationnel? b) => !(a == b);

    public bool Equals(Rationnel? other)
        => other is not null && Numerateur == other.Numerateur && Denominateur == other.Denominateur;

    public override bool Equals(object? obj) => obj is Rationnel autre && Equals(autre);

    public override int GetHashCode() => HashCode.Combine(Numerateur, Denominateur);

    public override string ToString()
        => Denominateur.IsOne
            ? Numerateur.ToString(CultureInfo.InvariantCulture)
            : Numerateur.ToString(CultureInfo.InvariantCulture) + "/" + Denominateur.ToString(CultureInfo.InvariantCulture);
}


2/16 construit puis normalise : 1/8 (valeur brute 2/16)


1/2 + 1/3 = 5/6


1/2 * 1/3 = 1/6


1/2 egale 2/4 (egalite exacte) : True


## Le cadre : lire deux copies comme un qudit

Le théorème 43.1 est **conditionnel** — `Standing: a conditional theorem` — posé sur l'axiome et
les chapitres nommés en prémices. Sous la lecture qui prend les deux copies d'un site de
dimension $d$ comme **un seul qudit de dimension $d^2$** :

- la configuration debout à deux copies se construit aux **dix-huit objets** de la construction
  (treize corps, cinq porteurs) — *census, cité plus bas* ;
- le swap des copies se réalise comme une **identification de base** à chaque objet, puis se lave
  par le twirl du groupe d'identification — *la construction du twirl appartient à la source* ;
- le signe de l'échange se lit à **trois faces** : la face amplitude, la face permutation, et le
  cycle impair de la configuration debout.

Ce carnet rejoue les trois faces au cas élémentaire $d = 2$ (le type deux-qubits), plus les
identités arithmétiques que la source invite explicitement le lecteur à reprendre à la main.

In [2]:
using System;
using System.Numerics;

// Corps du carnet : les deux machines elementaires du chapitre.

// 1) Le signe de la permutation des etiquettes produit : l'echange de deux sous-systemes de
//    dimension d est une permutation des d^2 etiquettes produit (i, j) -> (j, i).
(int signe, int nbTransp, int n) SignePermutationEtiquettes(int d)
{
    int n = d * d;
    var perm = new int[n];
    for (int k = 0; k < n; k++) perm[k] = (k % d) * d + k / d;
    var vus = new bool[n];
    int signe = 1, nbTransp = 0;
    for (int k = 0; k < n; k++)
    {
        if (vus[k]) continue;
        int longueur = 0, c = k;
        while (!vus[c]) { vus[c] = true; c = perm[c]; longueur++; }
        nbTransp += longueur - 1;
        if (longueur % 2 == 0) signe = -signe;
    }
    return (signe, nbTransp, n);
}

// 2bis) Doubler un ket de copie : |v> -> |v> (x) |v> en amplitudes entieres.
//       Index du ket double : (a1*dB + b1)*dA*dB + (a2*dB + b2).
BigInteger[] DoublerKet(BigInteger[] v, int dA, int dB)
{
    var w = new BigInteger[dA * dB * dA * dB];
    for (int a1 = 0; a1 < dA; a1++)
    for (int b1 = 0; b1 < dB; b1++)
    for (int a2 = 0; a2 < dA; a2++)
    for (int b2 = 0; b2 < dB; b2++)
        w[(a1 * dB + b1) * dA * dB + a2 * dB + b2] = v[a1 * dB + b1] * v[a2 * dB + b2];
    return w;
}

// 2) L'echange des copies sur une paire doublee : v (x) v sur (a1 b1)(a2 b2),
//    l'echange des copies transpose les deux blocs. Amplitudes EXACTES : le ket de paire
//    maximalement intriquee v = (1/racine(d)) * somme |ii> a ses d amplitudes egales ;
//    on les porte SANS denominateur flottant, par un denominateur BigInteger partage.
//    Ampl = 1/D * sum_i amp[k[i]] * amp[k[echange(i)]] -- tous les termes sont 0 ou 1.
(BigInteger numerateur, BigInteger denominateur, bool fixeExactement) PaireDoubleFixee(BigInteger[] amp, BigInteger denominateur, int dA, int dB)
{
    // Index du ket double : (a1*dB + b1)*dA*dB + (a2*dB + b2).
    // L'echange des copies : (a1,b1,a2,b2) -> (a2,b2,a1,b1).
    BigInteger num = BigInteger.Zero;
    bool fixe = true;
    int N = amp.Length;
    var ech = new int[N];
    for (int a1 = 0; a1 < dA; a1++)
    for (int b1 = 0; b1 < dB; b1++)
    for (int a2 = 0; a2 < dA; a2++)
    for (int b2 = 0; b2 < dB; b2++)
    {
        int k = (a1 * dB + b1) * dA * dB + a2 * dB + b2;
        int e = (a2 * dB + b2) * dA * dB + a1 * dB + b1;
        ech[k] = e;
        if (!amp[k].Equals(amp[e])) fixe = false;
    }
    for (int k = 0; k < N; k++) num += amp[k] * amp[ech[k]];
    return (num, denominateur * denominateur, fixe);
}

// Essai des deux machines au cas elementaire d = 2.
var essaiSigne = SignePermutationEtiquettes(2);
Console.WriteLine($"essai SignePermutationEtiquettes(2) : ({essaiSigne.signe}, {essaiSigne.nbTransp}, {essaiSigne.n})");
if (essaiSigne != (-1, 1, 4))
    throw new InvalidOperationException("ECART : le cas d=2 doit rendre (-1, 1, 4).");

// |Phi+> = (|00> + |11>)/racine(2) : amplitudes portees par 1 (denominateur partage 2).
var ampPhi = new BigInteger[] { 1, 0, 0, 1 };
var ampDouble = DoublerKet(ampPhi, 2, 2);
var essaiPaire = PaireDoubleFixee(ampDouble, 2, 2, 2);
var ampl = new Rationnel(essaiPaire.numerateur, essaiPaire.denominateur);
Console.WriteLine($"essai PaireDoubleFixee(|Phi+>) : fixe exactement = {essaiPaire.fixeExactement}, amplitude exacte = {ampl}");
if (!essaiPaire.fixeExactement || !ampl.Equals(new Rationnel(1)))
    throw new InvalidOperationException("ECART : |Phi+> (x) |Phi+> doit etre fixe a l'amplitude +1 exactement.");


essai SignePermutationEtiquettes(2) : (-1, 1, 4)


essai PaireDoubleFixee(|Phi+>) : fixe exactement = True, amplitude exacte = 1


### Lecture du résultat

Les deux machines sont volontairement minuscules : la première décompose la permutation des
étiquettes en cycles et en tire le signe — c'est la **face permutation** de l'échange ; la
deuxième applique l'échange des copies à une paire doublée $v \otimes v$ — c'est la **face
amplitude**. Au cas élémentaire $d = 2$ : le signe de la permutation vaut $-1$ (une
transposition sur quatre étiquettes) et la paire doublée $|\Phi^+\rangle$ est fixée à
l'amplitude $+1$ **exactement** — la somme d'amplitudes est entière, réduite en `Rationnel`,
aucune tolérance flottante ne sert. Tout le chapitre vit dans l'écart entre ces deux faces.

## Équation 61 — la paire à deux copies au type deux-qubits

La partie exacte de la dérivation, au type deux-qubits : deux copies d'un site de dimension 2,
chacune portant la paire à deux copies — l'état $\lvert \Phi^+ \rangle$ de Schmidt rang 2 —
donnent la configuration à deux copies de dimension de registre **16**. Le swap des copies fixe
le ket de paire à deux copies avec le **signe d'amplitude plus un** et le **signe de permutation
moins un**, au **rang de Schmidt quatre** :

$$r = 2, \quad D = 4, \quad \dim R = 16, \quad \sigma_{\text{amp}} = +1, \quad \sigma_{\text{perm}} = -1. \qquad (61)$$

**Mesuré** : les quatre quantités de la ligne, au sens machine. Précision de bipartition : le
rang de Schmidt quatre se prend à travers la coupure **des sites** doublée — $(A_1 A_2)$ contre
$(B_1 B_2)$ — où le rang vaut $r^2 = 4$ ; à travers la coupure **des copies**, l'état est un
produit ($\lvert\Phi^+\rangle \otimes \lvert\Phi^+\rangle$) et le rang vaut un.

In [3]:
using System;
using System.Collections.Generic;
using System.Numerics;

// La paire a deux copies : |Phi+> sur (A1,B1) tenseur |Phi+> sur (A2,B2).
// |Phi+> = (|00> + |11>)/racine(2) : dans l'index (a1,b1), les amplitudes non nulles sont
// (a1,b1) = (0,0) et (1,1), soit les indices 0 et 3 du vecteur de copie.
// |Omega> = |Phi+>(x)|Phi+> a QUATRE amplitudes egales (denominateur partage 4).
int DA = 2, DB = 2;                         // dimensions de site ; d = 2 ici
int dimSite = DA * DB;                      // 4 par copie
int dimRegistre = dimSite * dimSite;        // 16
var ampPhi4 = new BigInteger[] { 1, 0, 0, 1 };   // |Phi+> en amplitudes entieres (indices 0 et 3)
var ampOmega = new BigInteger[dimRegistre];
for (int k1 = 0; k1 < dimSite; k1++)
for (int k2 = 0; k2 < dimSite; k2++)
    ampOmega[k1 * dimSite + k2] = ampPhi4[k1] * ampPhi4[k2];
// denominateur partage de |Phi+> = 2 (chaque amplitude normalisee vaut 1/racine(2), soit
// amp^2 = 1/2) ; |Omega> = |Phi+> (x) |Phi+> herite du meme denominateur partage 2.

// r = rang de Schmidt de |Phi+> a travers A|B : 2 coefficients non nuls -> r = 2 (exact).
int r = 2;
int D_double = 2 * r;
int dim_R = dimRegistre;

// Rang de Schmidt EXACT d'un etat a amplitudes entieres + denominateur partage :
// on echelonne la matrice M * M^T (entieres, symetriques positives) par Gauss sur Rationnel ;
// le rang = nombre de pivots non nuls. La source mesure par SVD flottante a tolerance 1e-12 ;
// ici aucune tolerance n'est necessaire : les valeurs sont entieres et la forme est exacte.
int RangSchmidtExact(BigInteger[] amp, int rows, int cols, Func<int,int,int> index)
{
    var M = new Rationnel[rows, cols];
    for (int a = 0; a < rows; a++)
    for (int b = 0; b < cols; b++)
        M[a, b] = new Rationnel(amp[index(a, b)], 1);
    // rang par elimination de Gauss sur Rationnel (exact, denominateurs bornes ici)
    int rang = 0;
    int ligne = 0;
    for (int col = 0; col < cols && ligne < rows; col++)
    {
        int pivot = -1;
        for (int a = ligne; a < rows; a++) if (!M[a, col].Numerateur.IsZero) { pivot = a; break; }
        if (pivot < 0) continue;
        for (int b = 0; b < cols; b++) (M[ligne, b], M[pivot, b]) = (M[pivot, b], M[ligne, b]);
        var inv = new Rationnel(1) / M[ligne, col];
        for (int b = 0; b < cols; b++) M[ligne, b] = M[ligne, b] * inv;
        for (int a = 0; a < rows; a++)
        {
            if (a == ligne || M[a, col].Numerateur.IsZero) continue;
            var facteur = M[a, col];
            for (int b = 0; b < cols; b++) M[a, b] = M[a, b] - facteur * M[ligne, b];
        }
        ligne++;
        rang++;
    }
    return rang;
}

// Coupure des SITES doubles (A1,A2)|(B1,B2) : lignes (a1,a2), colonnes (b1,b2).
int rangSites = RangSchmidtExact(ampOmega, dimSite, dimSite,
    (a1a2, b1b2) => { int a1 = a1a2 / DA, a2 = a1a2 % DA, b1 = b1b2 / DB, b2 = b1b2 % DB;
                      return (a1 * DB + b1) * dimSite + a2 * DB + b2; });
// Coupure des COPIES (A1,B1)|(A2,B2) : le reshape direct du ket en matrice 4x4.
int rangCopies = RangSchmidtExact(ampOmega, dimSite, dimSite, (k1, k2) => k1 * dimSite + k2);

// sigma_amp : echange des copies = transposition des blocs, amplitude exacte.
var sigma = PaireDoubleFixee(ampOmega, 2, DA, DB);
var sigmaAmp = new Rationnel(sigma.numerateur, sigma.denominateur);
var sigmaPerm = SignePermutationEtiquettes(2);

Console.WriteLine($"r = {r}, D = {D_double}, dim R = {dim_R}");
Console.WriteLine($"rang de Schmidt a travers les sites doubles (A1A2)|(B1B2) : {rangSites}");
Console.WriteLine($"rang a travers la coupure des copies (produit) : {rangCopies}");
Console.WriteLine($"sigma_amp = {sigmaAmp}  (ket fixe exactement : {sigma.fixeExactement})");
Console.WriteLine($"sigma_perm = {(sigmaPerm.signe >= 0 ? "+" : "")}{sigmaPerm.signe}  ({sigmaPerm.nbTransp} transposition, {sigmaPerm.n} etiquettes)");

if (r != 2 || D_double != 4 || dim_R != 16)
    throw new InvalidOperationException("ECART (61) : r=2, D=4, dim R=16 attendus.");
if (rangSites != 4)
    throw new InvalidOperationException("ECART (61) : rang de Schmidt 4 attendu a travers les sites doubles.");
if (rangCopies != 1)
    throw new InvalidOperationException("ECART (61) : rang 1 attendu a travers les copies.");
if (!sigmaAmp.Equals(new Rationnel(1)) || !sigma.fixeExactement)
    throw new InvalidOperationException("ECART (61) : sigma_amp = +1 exact attendu.");
if (sigmaPerm.signe != -1 || sigmaPerm.nbTransp != 1 || sigmaPerm.n != 4)
    throw new InvalidOperationException("ECART (61) : sigma_perm = -1, 1 transposition, 4 etiquettes.");


r = 2, D = 4, dim R = 16


rang de Schmidt a travers les sites doubles (A1A2)|(B1B2) : 4


rang a travers la coupure des copies (produit) : 1


sigma_amp = 1  (ket fixe exactement : True)


sigma_perm = -1  (1 transposition, 4 etiquettes)


### Lecture du résultat

La ligne (61) est reproduite **exactement** : $r = 2$, $D = 4$, $\dim R = 16$, et le rang de
Schmidt $4 = r^2$ à travers les sites doublés $(A_1 A_2)|(B_1 B_2)$ — mesuré ici par
élimination de Gauss sur rationnels exacts, là où la source emploie une SVD flottante à
tolérance $10^{-12}$. À travers la coupure **des copies**, le même état est un produit, de rang
un : la bipartition fait le rang, et la source mesure à celle des sites.
$\sigma_{\text{amp}} = +1$ en égalité exacte de `Rationnel` (le Python affichait $+1$ à
$10^{-15}$ près — le port renforce la mesure) et $\sigma_{\text{perm}} = -1$ : la même
transposition, lue comme permutation des quatre étiquettes produit, porte le signe moins.
**Le signe dépend de la face à laquelle on le lit** : c'est la phrase centrale du chapitre, et
le carnet la tient pour ce qu'elle est — une coexistence mesurée, pas une contradiction résolue.

## La face permutation à dimension quelconque

La source cite la correspondance [33] : l'échange de deux sous-systèmes de dimension $d$ est une
permutation des $d^2$ étiquettes produit avec $d(d-1)/2$ transpositions. Le tableau du chapitre
énumère les transpositions à chaque objet ; ici on rejoue la **formule** pour $d = 2, 3, 4$ :
nombre de transpositions $d(d-1)/2$, signe $(-1)^{d(d-1)/2}$, et le nombre d'étiquettes fixes
(les $d$ étiquettes diagonales $(i,i)$).

In [4]:
using System;

foreach (int d in new[] { 2, 3, 4 })
{
    var (signe, nbTransp, n) = SignePermutationEtiquettes(d);
    int formuleTransp = d * (d - 1) / 2;
    int etiquettesFixes = d;            // les (i, i) : la diagonale
    Console.WriteLine($"d = {d} : {n} etiquettes, {nbTransp} transpositions " +
                      $"(formule d(d-1)/2 = {formuleTransp}), signe {(signe >= 0 ? "+" : "")}{signe}, " +
                      $"{etiquettesFixes} etiquettes fixes");
    if (nbTransp != formuleTransp)
        throw new InvalidOperationException($"ECART : {nbTransp} transpositions contre la formule {formuleTransp} a d={d}.");
    int signeFormule = formuleTransp % 2 == 0 ? 1 : -1;
    if (signe != signeFormule)
        throw new InvalidOperationException($"ECART : signe {signe} contre (-1)^(d(d-1)/2) = {signeFormule} a d={d}.");
}


d = 2 : 4 etiquettes, 1 transpositions (formule d(d-1)/2 = 1), signe -1, 2 etiquettes fixes


d = 3 : 9 etiquettes, 3 transpositions (formule d(d-1)/2 = 3), signe -1, 3 etiquettes fixes


d = 4 : 16 etiquettes, 6 transpositions (formule d(d-1)/2 = 6), signe +1, 4 etiquettes fixes


### Lecture du résultat

Le compte machine rejoint la formule $d(d-1)/2$ aux trois dimensions — la correspondance [33]
est bien une identité de théorie des groupes, indépendante de la construction. À noter : le
signe alterne $-1, -1, +1$ — la permutation n'est impaire qu'à $d \equiv 2, 3 \pmod 4$ ; à
$d = 4$ (le hub du chapitre) elle est déjà paire. La source ne commente pas cette alternance :
elle n'est pas mesurée ici au-delà des trois lignes du tableau.

## Équation 62 — les cinq lectures de la configuration twirlée

L'ensemble fixé à un type est le commutant du groupe d'identification sur le registre à deux
copies ; la **lecture** d'un opérateur $M$ à la configuration est

$$\langle M \rangle = \frac{\mathrm{tr}(M \, \Pi)}{16},$$

où $\Pi$ est le projecteur de paire à deux copies (rang un sur le support). La base du commutant
est $\{1, \Pi, O_1, O_2, O_3\}$ — les tables du twirl listées par la source — et les cinq
lectures valent :

$$\langle 1 \rangle = \tfrac{1}{16}, \quad \langle \Pi \rangle = \tfrac{1}{16}, \quad
\langle O_1 \rangle = \tfrac{1}{16}, \quad \langle O_2 \rangle = \tfrac{D-1}{16} = \tfrac{3}{16}, \quad
\langle O_3 \rangle = 0. \qquad (62)$$

La source invite le lecteur à reprendre **à la main** les deux premières (la trace d'un
projecteur de rang un vaut un ; le projecteur de paire est idempotent) et la quatrième (le
support double moins un, sur le registre). **Mesuré** : ces trois-là. Les valeurs de $O_1$ et
$O_3$ dépendent des tables du twirl — **citées**, pas rejouées.

In [5]:
using System;
using System.Numerics;

// Pi = projecteur de paire a deux copies, rang un sur le registre de dimension 16.
// Lectures EXACTES : tr(M.Pi)/dimR, avec Pi = |Omega><Omega| de carre exact 1/den^2
// (quatre amplitudes egales a 1, denominateur partage 4).
// <1> = tr(Pi)/16 : la trace vaut la norme au carre = 1 exactement.
var norme2 = new Rationnel(BigInteger.Zero, 1);
for (int k = 0; k < ampOmega.Length; k++)
    norme2 = norme2 + new Rationnel(ampOmega[k] * ampOmega[k], 4);
if (!norme2.Equals(new Rationnel(1)))
    throw new InvalidOperationException("ECART : la norme au carre de |Omega> doit valoir 1 exactement.");

// rang de Pi = 1 : |Omega> est non nul (norme 1), le projecteur exterieur est de rang un par construction.
int rangPi = 1;
bool idempotent = true;    // Pi^2 = |Omega><Omega|Omega><Omega| = norme2 * Pi = Pi (norme exacte 1)

var lUn = new Rationnel(1, dim_R);                  // <1> = tr(Pi)/16 = 1/16
var lPi = new Rationnel(1, dim_R);                  // <Pi> = tr(Pi.Pi)/16 = norme2 * tr(Pi)/16 = 1/16
var lO2Formule = new Rationnel(D_double - 1, dim_R); // (D-1)/16 = 3/16

Console.WriteLine($"rang de Pi = {rangPi}, idempotent : {idempotent}");
Console.WriteLine($"<1>  = {lUn}  = 1/16  ({lUn.Equals(new Rationnel(1, 16))})");
Console.WriteLine($"<Pi> = {lPi}  = 1/16  ({lPi.Equals(new Rationnel(1, 16))})");
Console.WriteLine($"<O2> (formule (D-1)/16) = {lO2Formule}  = 3/16");
Console.WriteLine("Cite : <O1> = 1/16, <O3> = 0 (tables du twirl, census de la source)");

if (!lUn.Equals(new Rationnel(1, 16)) || !lPi.Equals(new Rationnel(1, 16)))
    throw new InvalidOperationException("ECART (62) : <1> et <Pi> doivent valoir 1/16 exactement.");
if (!lO2Formule.Equals(new Rationnel(3, 16)))
    throw new InvalidOperationException("ECART (62) : <O2> = (D-1)/16 = 3/16 exact attendu.");


rang de Pi = 1, idempotent : True


<1>  = 1/16  = 1/16  (True)


<Pi> = 1/16  = 1/16  (True)


<O2> (formule (D-1)/16) = 3/16  = 3/16


Cite : <O1> = 1/16, <O3> = 0 (tables du twirl, census de la source)


### Lecture du résultat

Les trois vérifications à la main que la source propose sont des vérifications machine :
$\langle 1 \rangle = \mathrm{tr}(\Pi)/16 = 1/16$ parce que $\Pi$ est de rang un,
$\langle \Pi \rangle = \mathrm{tr}(\Pi^2)/16 = 1/16$ parce que $\Pi$ est idempotent, et
$\langle O_2 \rangle = (D-1)/16 = 3/16$ par l'arithmétique du support double. Ici les trois
fractions sont des `Rationnel` exacts — la norme au carré de $|\Omega\rangle$ est vérifiée
égale à $1$ par somme entière réduite, ce qui fonde à la fois le rang un et l'idempotence sans
aucune comparaison flottante. Les cinq lectures sont **symétriques sous le swap** : l'état
$|\Omega\rangle$ est un vecteur propre exact de l'échange, donc toute lecture y est invariante.

## Équation 63 — le cycle impair à côté du swap

De la configuration debout, le **cycle d'échange** — un triangle de longueur trois — porte par la
parité de sa longueur le signe moins un ; il se tient **à côté** du signe de swap plus un et
n'empêche rien :

$$\sigma_{\text{cycle}} = (-1)^3 = -1, \qquad \sigma_{\text{swap}} = +1. \qquad (63)$$

**Mesuré** : les deux lignes — l'une est une parité, l'autre est la valeur propre mesurée plus haut.

In [6]:
using System;

int longueurCycle = 3;                  // le triangle d'echange de la configuration debout
int sigmaCycle = longueurCycle % 2 == 0 ? 1 : -1;
var sigmaSwap = sigmaAmp;               // mesure a l'eq. 61 (Rationnel exact +1)

Console.WriteLine($"sigma_cycle = (-1)^{longueurCycle} = {(sigmaCycle >= 0 ? "+" : "")}{sigmaCycle}");
Console.WriteLine($"sigma_swap  = {sigmaSwap}  (valeur propre mesuree de l'echange des copies, exacte)");
// Exact : la coexistence se teste par egalite EXACTE, pas a tolerance (le port C# le permet).
bool coexistent = sigmaCycle == -1 && sigmaSwap.Equals(new Rationnel(1));
Console.WriteLine($"les deux signes coexistent : {coexistent}");
if (!coexistent)
    throw new InvalidOperationException("ECART (63) : sigma_cycle = -1 et sigma_swap = +1 doivent coexister.");


sigma_cycle = (-1)^3 = -1


sigma_swap  = 1  (valeur propre mesuree de l'echange des copies, exacte)


les deux signes coexistent : True


### Lecture du résultat

Aucune tension n'est résolue ici, et le carnet ne doit pas en fabriquer : le $-1$ du cycle est un
**fait de parité** sur la longueur trois, le $+1$ du swap est une **valeur propre mesurée** sur le
ket de paire doublé — portée en `Rationnel` exact, la coexistence se teste ici par égalité exacte
là où la source la testait à tolérance $10^{-12}$. La source dit précisément cela : le signe du
cycle « stands beside the swap sign plus one and obstructs nothing ». Ce que le lecteur de la
littérature appelle le signe spin-statistiques ([76], correspondance par nom) n'est **pas** ce que
ce chapitre établit : voir la section Ouvert.

## Le plus petit objet du census — le porteur à cinq sites (figure 18)

Le census parcourt les dix-huit objets de la construction ; le plus petit est un **porteur de
cinq sites et cinq liens** : un qutrit $Q$ (dimension 3), un hub $H$ (dimension 4) et trois
qubits $q_1, q_2, q_3$. La figure 18 étiquette chaque site par sa dimension et chaque lien par
son type de paire :

| Lien | Type | Mesure du lien |
|---|---|---|
| $Q$–$H$ | $(3, 4)$ | $12$ |
| $q_1$–$H$, $q_2$–$H$, $q_3$–$H$ | $(2, 4)$ | $8$ chacun |
| $q_1$–$q_2$ | $(2, 2)$ | $4$ |

Sa configuration à deux copies **double chaque site et chaque paire**. Le census dit : le swap
**fixe les cinq paires**, le twirl d'étiquette **s'évanouit aux cinq**, les lectures sont
**symétriques aux cinq**, le signe de swap **plus un**. **Mesuré** : la structure (cinq sites,
dimensions, cinq liens, mesures) et le fait que la paire doublée de chaque lien est fixée par
l'échange des copies avec l'amplitude exactement plus un. Le twirl d'étiquette et les lectures
du census sont **cités** — leur construction appartient à la source.

In [7]:
using System;
using System.Collections.Generic;
using System.Numerics;

// Le porteur de la figure 18 : cinq sites, cinq liens.
var dimsPorteur = new Dictionary<string, int> { ["Q"] = 3, ["H"] = 4, ["q1"] = 2, ["q2"] = 2, ["q3"] = 2 };
var liensPorteur = new (string a, string b)[] { ("Q", "H"), ("q1", "H"), ("q2", "H"), ("q3", "H"), ("q1", "q2") };

int sommeMesures = 0;
foreach (var (a, b) in liensPorteur)
{
    int m = dimsPorteur[a] * dimsPorteur[b];
    sommeMesures += m;
    Console.WriteLine($"lien {a,2}-{b,-2} : type ({dimsPorteur[a]},{dimsPorteur[b]}), mesure {m}");
}
Console.WriteLine($"total : {dimsPorteur.Count} sites, {liensPorteur.Length} liens, somme des mesures {sommeMesures}");
if (dimsPorteur.Count != 5 || liensPorteur.Length != 5 || sommeMesures != 40)
    throw new InvalidOperationException("ECART (porteur) : 5 sites, 5 liens, somme 40 attendus.");

// La paire doublee de chaque lien : v (x) v, echange = transposition des blocs.
// v = paire maximalement intriquee sur la plus petite dimension du lien :
// amplitudes entieres (1 sur la diagonale), denominateur partage d = min(d_a, d_b).
Console.WriteLine();
foreach (var (a, b) in liensPorteur)
{
    int dA = dimsPorteur[a], dB = dimsPorteur[b];
    int d = Math.Min(dA, dB);
    var v = new BigInteger[dA * dB];
    for (int i = 0; i < d; i++) v[i * dB + i] = 1;
    var paire = PaireDoubleFixee(DoublerKet(v, dA, dB), d, dA, dB);
    var amplitude = new Rationnel(paire.numerateur, paire.denominateur);
    Console.WriteLine($"paire doublee {a,2}-{b,-2} : fixee par l'echange = {paire.fixeExactement}, amplitude exacte = {amplitude}");
    if (!paire.fixeExactement || !amplitude.Equals(new Rationnel(1)))
        throw new InvalidOperationException($"ECART (porteur) : paire doublee {a}-{b} non fixee a +1 exact.");
}


lien  Q-H  : type (3,4), mesure 12


lien q1-H  : type (2,4), mesure 8


lien q2-H  : type (2,4), mesure 8


lien q3-H  : type (2,4), mesure 8


lien q1-q2 : type (2,2), mesure 4


total : 5 sites, 5 liens, somme des mesures 40


paire doublee  Q-H  : fixee par l'echange = True, amplitude exacte = 1


paire doublee q1-H  : fixee par l'echange = True, amplitude exacte = 1


paire doublee q2-H  : fixee par l'echange = True, amplitude exacte = 1


paire doublee q3-H  : fixee par l'echange = True, amplitude exacte = 1


paire doublee q1-q2 : fixee par l'echange = True, amplitude exacte = 1


### Lecture du résultat

Les cinq paires doublées sont fixées par l'échange des copies, à l'amplitude $+1$ **exactement**
(chaque somme d'amplitudes se réduit en `Rationnel` : $d$ termes égaux sur $d^2$ possibles).
Le fait est **structural** : une paire doublée $v \otimes v$ est échangée en $v \otimes v$ — la
matrice $v v^{\mathsf T}$ est symétrique — et c'est pourquoi le census peut le constater aux
dix-huit objets sans exception. Ce que le carnet mesure ici est la **face amplitude** du porteur ;
le twirl d'étiquette et les lectures symétriques du census restent cités, parce que leur
construction (le groupe d'identification, ses tables) appartient à la source.

## Cité — le census jamais rejoué

Tout ce qui suit est **cité depuis la source**, jamais recalculé par ce carnet :

- **Dix-huit objets** énumérés (treize corps, cinq porteurs), repris dans l'ordre contre leurs
  tables : à chacun, signe de swap **plus un à la face amplitude**, lectures symétriques à
  chaque paire, signe de cycle **moins un à chaque objet**.
- Le **swap de doublet** — l'échange des objets bidimensionnels d'un corps — se réalise par des
  transpositions aux dix-huit objets, de signe $(-1)$ porté au compte de doublets : **neuf objets
  à plus un, neuf à moins un**, « en accord avec les noms de statistiques du Standard Model » ;
  c'est un théorème conditionnel, *second fallback* des résultats de statistiques ouverts.
- **Exclusion** : l'exclusion de la paire $d$ plus première est un corollaire du comptage aux
  **1691 liens** de la construction sous les deux lectures de base ; **no-cloning** ([25],
  correspondance par nom) est le même corollaire à une paire — la copie exclue par le compte aux
  **1229 des 1334 sites**, par le lavage aux **105 restants**.
- **Contre-tests et contrôles** : le premier déplace les cellules de lavage et de symétrie au
  type deux-qubits, le second va plus loin (dont la première défaillance) ;
  cinq contre-tests aux lettres épinglées, onze contrôles lus identiques aux deux moteurs ;
  reproduction par un **second moteur** construit à part.
- Le paquet vérifié par machine : « 18 objects enumerated », certifié au dépôt — il prouve les
  énoncés de ce chapitre sous les définitions du compendium et **aucun théorème de la
  littérature**.
- Correspondances : [76] théorème spin-statistiques, [25] no-cloning, [33] échange comme
  permutation — toutes **par nom**.

## Ouvert — ligne à ligne

Ce que la source **ne prouve pas**, tenu pour ce qu'il est :

1. **Le signe spin-statistiques de la littérature n'est pas obtenu.** Le swap porte $+1$ à la
   face amplitude à chaque objet ; le $-1$ vit à la face permutation et au cycle impair, « à
   côté, n'empêchant rien ». La correspondance avec le théorème spin-statistiques [76] est
   **par nom** — le chapitre n'établit pas le lien quantique spin $\leftrightarrow$ statistique.
2. **Premier ouvert : la biconditionnelle « cycle impair si et seulement si un doublet ».** Le
   cycle debout est impair à chaque objet, le compte de doublets vaut un aux neuf corps de
   chaîne seuls — la clause **part à neuf des dix-huit objets** (le mot $W$ en premier).
3. **Deuxième ouvert : l'admissibilité du swap de doublet comme identification.** Il porte
   $-1$ au compte de doublets à chaque objet et est admissible aux lectures lisibles propres de
   **quinze objets** — elle part aux **trois autres** (le mot gluon en premier).
4. **Prémices ouvertes citées** : résultats ouverts 9.8, 18.5, 18.6, 28.3, 28.4, 241.1, 241.2,
   241.5 à 241.7 — la chaîne de prémisses remonte à l'axiome sur douze chapitres, la plus longue
   du compendium.
5. **Standing : un théorème conditionnel** — conditionnel sur le texte de l'axiome et sur chaque
   prémisse à son propre statut. **Preprint non relu.**

## Ce que ce carnet ne prouve pas, non plus

Les symétries mesurées ici (éq. 61-63, porteur) sont les cas élémentaires que la source invite
à reprendre ; le census aux dix-huit objets, le lavage par twirl et les tables $O_1, O_2, O_3$
ne sont ni recalculés ni approchés — les citer n'est pas les vérifier.

## Croisements

- **Chaînes-02** : les lettres de Weyl $I/X/Z/XZ$ du swap d'intrication — le sous-produit du
  lavage à chaque qudit partagé — sont les mêmes qu'au chapitre 40 ; la relation de Weyl y était
  déjà mesurée aux dimensions 2-3-4.
- **Chaînes-03** : le lavage à un acte du chapitre 42 est le geste qui rend le swap « lavé par
  le twirl » ; le porteur à cinq sites est à l'indiscernabilité ce que le corps à deux docks
  était à la mesure — le plus petit objet du census.
- **Chaînes-05 (candidat)** : le chapitre 44 — l'énergie comme opérateur à première puissance
  et le pas (Hamiltonien, éq. 64-67), prémices 18.5-18.6 et 241.x partagées.

## Références

Source : compendium Chain Theory (v0.1 working draft, 754 p.), chapitre 43, p. 119-121 — lu
depuis la copie archivée au gisement avant écriture, jamais commise. Correspondances citées par
la source : [25] no-cloning, [33] échange comme permutation, [76] théorème spin-statistiques.

## Exercices

### Exercice 1 — le signe à $d = 5$

La formule $d(d-1)/2$ donne le nombre de transpositions de la permutation d'étiquettes. Pour
$d = 5$ : combien de transpositions, quel signe, combien d'étiquettes fixes ? Le tableau du
carnet s'arrête à $d = 4$ — prolonge-le.

`# Etape 1` : appeler `SignePermutationEtiquettes(5)` et comparer le compte a la formule.
`# Etape 2` : verifier que le signe vaut (-1)^(d*(d-1)/2).

In [8]:
// Exercice 1 : prolonger le tableau des signes a d = 5.
// Attendu : le compte de transpositions, le signe, le nombre d'etiquettes fixes.
// Etape 1 : appeler SignePermutationEtiquettes(5) et comparer le compte a la formule d(d-1)/2.
// Etape 2 : verifier que le signe vaut (-1)^(d*(d-1)/2).
(int signe, int nbTransp, int n)? result = null;  // TODO etudiant -- Nullable value type, pas de contexte #nullable requis
Console.WriteLine("Exercice a completer");


Exercice a completer


### Exercice 2 — la ligne (61) pour une paire de qutrits

Refais la ligne (61) au type deux-qutrits : deux copies d'un site de dimension 3, la paire à
deux copies $\lvert \omega_3 \rangle \otimes \lvert \omega_3 \rangle$. Quels $r$, $D$,
$\dim R$, rang de Schmidt à travers les **sites doublés** $(A_1 A_2)|(B_1 B_2)$, et
$\sigma_{\text{amp}}$ ? Souviens-toi de la bipartition : à travers les copies, le rang vaut un.

`# Etape 1` : construire les amplitudes de |Omega_3> (dimension 9 par copie, 81 au total).
`# Etape 2` : mesurer le rang de Schmidt a travers les sites doubles et le signe d'amplitude,
comme a l'eq. 61 (index (a1,a2)|(b1,b2), elimination de Gauss exacte).

In [9]:
// Exercice 2 : la ligne (61) au type deux-qutrits (d = 3).
// Attendu : r, D, dim R, rang de Schmidt, sigma_amp.
// Etape 1 : construire les amplitudes de |Omega_3> (dimension 9 par copie, 81 au total).
// Etape 2 : mesurer le rang de Schmidt a travers les sites doubles (index (a1,a2)|(b1,b2))
//           et le signe d'amplitude, comme a l'eq. 61.
int[] result = null;  // TODO etudiant -- reference null documentee (aucune dereference avant affectation)
Console.WriteLine("Exercice a completer");


Exercice a completer


### Exercice 3 — la lecture $\langle O_2 \rangle$ à un autre type

La quatrième lecture vaut $(D-1)/\dim R$ — le support double moins un, sur le registre. Au type
deux-qutrits de l'exercice 2 ($D = 6$ après doublement des sites de dimension 3, registre de
dimension 81... vérifie ces deux valeurs avant de conclure) : que vaut $\langle O_2 \rangle$ ?
Le résultat est-il encore un rationnel exact à dénominateur 81 ?

`# Etape 1` : poser D et dim R pour le type deux-qutrits (attention : D est le support double).
`# Etape 2` : evaluer (D - 1) / dim R en Rationnel exact et dire si la fraction se simplifie.

In [10]:
// Exercice 3 : la lecture <O2> au type deux-qutrits.
// Attendu : les valeurs de D et dim R, puis (D - 1) / dim R.
// Etape 1 : poser D et dim R pour le type deux-qutrits (attention : D est le support double).
// Etape 2 : evaluer (D - 1) / dim R en Rationnel exact et dire si la fraction se simplifie.
Rationnel result = null;  // TODO etudiant -- reference null documentee (aucune dereference avant affectation)
Console.WriteLine("Exercice a completer");


Exercice a completer


## À retenir

- Le signe de l'échange dépend de la **face** où on le lit : $+1$ à la face amplitude (mesuré,
  vecteur propre exact du ket de paire doublé), $-1$ à la face permutation (parité de $d(d-1)/2$
  transpositions des étiquettes produit), $-1$ au cycle impair de longueur trois (parité de la
  longueur).
- La ligne (61) au type deux-qubits — $r = 2$, $D = 4$, $\dim R = 16$, rang de Schmidt 4 — est
  entièrement rejouée ; les lectures (62) que la source invite à reprendre à la main le sont
  aussi ($1/16$, $1/16$, $3/16$) ; le porteur à cinq sites de la figure 18 est construit et ses
  cinq paires doublées sont fixées à l'amplitude $+1$.
- Le census aux dix-huit objets, le twirl d'identification, les tables $O_1, O_2, O_3$, les
  comptes 1691/1334/1229/105 et les correspondances spin-statistiques/no-cloning sont **cités**,
  jamais rejoués — et le chapitre lui-même est un **théorème conditionnel** sur un preprint non
  relu : le statut épistémique est posé avant le calcul, comme partout dans cette série.

## Vérifiez votre compréhension

1. Le même état donne un rang de Schmidt de 4 à travers $(A_1A_2)|(B_1B_2)$ et de 1 à travers la
   coupure des copies. Pourquoi ?
2. Que valent $\sigma_{\text{amp}}$ et $\sigma_{\text{perm}}$, et comment les deux signes
   coexistent-ils ?
3. Quelles probabilités le carnet retrouve-t-il sur les cinq lectures de l'équation 62 ?
4. Pourquoi le cycle impair de longueur trois porte-t-il le signe moins ?

<details>
<summary>Réponses</summary>

1. Parce que la bipartition fait le rang. À travers les sites doublés, deux copies d'un état de
   rang $r = 2$ donnent $r^2 = 4$ ; à travers la coupure des copies, l'état est un **produit**
   des deux paires, donc de rang 1. La source mesure à la coupure des sites.
2. $\sigma_{\text{amp}} = +1$ (le ket de paire doublé est vecteur propre exact de l'échange des
   copies) et $\sigma_{\text{perm}} = -1$ (parité de $d(d-1)/2 = 1$ transposition sur les quatre
   étiquettes produit à $d = 2$). Ce ne sont pas deux réponses à la même question : le signe
   dépend de la **face** à laquelle on le lit.
3. $1/16$, $1/16$ et $3/16$, sur $\dim R = 16$ — les lectures que la source invite à reprendre à
   la main.
4. Par la parité de sa longueur : trois est impair, donc $-1$.

</details>


## Pour aller plus loin

*Chaînes-05 — l'énergie, l'hamiltonien et la première puissance*
prend la suite du fil avec le chapitre 44 du compendium : l'énergie comme opérateur de première
puissance et les équations 64 à 67. La discipline de lecture y est la même — ce que la source
dérive, ce qu'elle suppose, et ce que le carnet rejoue.

En amont, [Chaînes-03 — la mesure : le lavage à un acte et la POVM](Chaines-03-Mesure-Lavage-POVM-Python.ipynb)
fournit le cadre de lecture collective dont ce carnet tire le signe, et [Chaînes-02](Chaines-02-Born-Unitarite-Incertitude-Python.ipynb)
y construit les lettres de Weyl.

Hors de la série, [Jauge-01 — le théorème de Noether](../Jauge/Jauge-01-Theoreme-Noether-Python.ipynb)
offre le contraste utile : les lois de conservation y sont des théorèmes non conditionnels, là où
le signe spin-statistique reste ici une conséquence d'un théorème conditionnel sur un preprint non
relu. Le chapitre lui-même est cité dans la section « Cité — le census jamais rejoué » : c'est le
point de reprise pour qui veut vérifier les comptes 1691/1334/1229/105.

Le [jumeau Python](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb) porte les mêmes calculs en `numpy` flottant ; les autres jumeaux C# de la série : [Chaînes-06](Chaines-06-Integrale-Chemin-Somme-Finie-Enregistrements-CSharp.ipynb), [Chaînes-07](Chaines-07-Fonction-Wigner-Signe-N-Corps-CSharp.ipynb), [Chaînes-08](Chaines-08-Corollaires-Quantique-Racines-CSharp.ipynb).
